# 챕터 3 — 양방향 무역수지: 어느 나라와 흑자, 어느 나라와 적자인가 (2007–2025)

한국이 **상대국별로** 흑자를 보는가 적자를 보는가, 그 구도가 20년간 어떻게 변했는지를 인과 해석 없이 서술하는 기술통계 노트북이다. 챕터 1이 전체 무역수지를 봤다면, 여기서는 그것을 교역 상대국별로 쪼갠다. 함께 있는 문서 `chapter03.md`가 이 결과를 이야기로 엮은 것이다.

## 이 데이터베이스에 대하여 (출처·집계 기준)

이 데이터베이스는 관세청이 OpenAPI(품목별 국가별 수출입실적(GW), https://www.data.go.kr/data/15100475/openapi.do)로 공개하는 월별 수출입 통계를 **2007년 1월부터 2026년 3월까지** 한데 모아 하나의 파일로 만든 것이다.

**수치의 집계 기준(관세청 정의).** 수출입 신고 통관 자료를 국가 및 HS Code(2·4·6·10단위)별로 집계한 국가별 품목별 무역통계다. 금액은 미화(USD)이며, 수출은 FOB(신고금액), 수입은 CIF(과세가격) 기준이다. 중량은 순중량(kg). 국가는 수출은 최종목적국, 수입은 원산국을 원칙으로 하며 무역통계부호상 ISO 코드로 분류한다. 단순 통과물품이나 일시 반입·반출 물품은 제외된다(물적 자원의 증감이 없으므로). 통계는 매월 수출입 신고의 정정·취하를 반영해 전월까지 자료를 현행화한다(주기 1개월).

## 0. 규칙과 함정

- 수지 = 수출(FOB) − 수입(CIF), 미화 달러. 금액은 **억 달러**(=100M USD)로 표기.
- **2026 제외**(부분년).
- **집계 코드(EU 등) 제외** — 개별국과 겹침.
- **국가 기준 함정**: 수출=최종목적국·수입=원산국. 중계무역(홍콩)·편의치적(마셜제도)이 특정국 수지를 부풀린다(6절).

In [ ]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

EN = {'미합중국':'USA','중국':'China','일본':'Japan','호주':'Australia','베트남':'Vietnam',
      '홍콩':'HK','인도':'India','싱가포르':'Singapore','대만':'Taiwan','필리핀':'Philippines',
      '폴란드':'Poland','사우디아라비아':'Saudi','카타르':'Qatar','아랍에미리트':'UAE',
      '이라크':'Iraq','독일':'Germany','쿠웨이트':'Kuwait','인도네시아':'Indonesia'}
def L(k): return EN.get(k, k)

DB_PATH = os.path.join("data", "processed", "kcsdb.duckdb")
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DB 없음: {DB_PATH} — Releases에서 받아 data/processed/ 에 배치")
con = duckdb.connect(DB_PATH, read_only=True)
def q(sql): return con.sql(sql).df()
print("연결 완료 —", f"{con.sql('SELECT COUNT(*) FROM fact_trade').fetchone()[0]:,}", "거래행")

## 3. 2025년 — 흑자국과 적자국

In [ ]:
# 2025 흑자 상위 10 (억달러)
surplus = q('''
    SELECT COALESCE(c.name_ko_mofa,c.name_ko_kcs) AS 국가,
           ROUND(SUM(f.exp_dlr-f.imp_dlr)/1e8,0) AS 수지_억,
           ROUND(SUM(f.exp_dlr)/1e8,0) AS 수출_억, ROUND(SUM(f.imp_dlr)/1e8,0) AS 수입_억
    FROM fact_trade f JOIN dim_country c USING(stat_cd)
    WHERE f.yyyymm//100=2025 AND c.stat_cd<>'EU'
    GROUP BY 1 ORDER BY 수지_억 DESC LIMIT 10
''')
deficit = q('''
    SELECT COALESCE(c.name_ko_mofa,c.name_ko_kcs) AS 국가,
           ROUND(SUM(f.exp_dlr-f.imp_dlr)/1e8,0) AS 수지_억,
           ROUND(SUM(f.exp_dlr)/1e8,0) AS 수출_억, ROUND(SUM(f.imp_dlr)/1e8,0) AS 수입_억
    FROM fact_trade f JOIN dim_country c USING(stat_cd)
    WHERE f.yyyymm//100=2025 AND c.stat_cd<>'EU'
    GROUP BY 1 ORDER BY 수지_억 ASC LIMIT 10
''')
print("흑자 상위:"); print(surplus.to_string(index=False))
print("\n적자 상위:"); print(deficit.to_string(index=False))

# 발산 막대차트 (흑자 초록 / 적자 빨강)
s = surplus.head(8)[::-1]; d = deficit.head(8)
labels = [L(x) for x in s['국가']] + [L(x) for x in d['국가']]
vals = list(s['수지_억']) + list(d['수지_억'])
colors = ['seagreen' if v>0 else 'crimson' for v in vals]
f=plt.figure(figsize=(8,4.6)); ax=f.gca()
ax.barh(range(len(vals)), vals, color=colors)
ax.set_yticks(range(len(vals))); ax.set_yticklabels(labels, fontsize=7)
ax.axvline(0,color='black',lw=.8); ax.set_xlabel('Trade balance 2025 (100M USD)')
ax.set_title('Bilateral Trade Balance, Korea 2025 (surplus vs deficit)')
ax.grid(alpha=.3, axis='x'); f.tight_layout(); plt.show()

In [ ]:
# 흑자국 vs 적자국 수·합계 (2025)
q('''
    WITH b AS (SELECT c.stat_cd, SUM(f.exp_dlr-f.imp_dlr) bal
               FROM fact_trade f JOIN dim_country c USING(stat_cd)
               WHERE f.yyyymm//100=2025 AND c.stat_cd<>'EU' GROUP BY 1)
    SELECT COUNT(*) FILTER (WHERE bal>0) AS 흑자국수,
           COUNT(*) FILTER (WHERE bal<0) AS 적자국수,
           ROUND(SUM(bal) FILTER (WHERE bal>0)/1e8,0) AS 흑자합_억,
           ROUND(SUM(bal) FILTER (WHERE bal<0)/1e8,0) AS 적자합_억,
           ROUND(SUM(bal)/1e8,0) AS 순수지_억
    FROM b
''')

## 5. 20년간 변한 것 — 대중국 수지의 반전

In [ ]:
# 주요 파트너 수지 추이 (억달러)
traj = q('''
    SELECT f.yyyymm//100 yr, c.name_ko_mofa 국가, ROUND(SUM(f.exp_dlr-f.imp_dlr)/1e8,0) bal
    FROM fact_trade f JOIN dim_country c USING(stat_cd)
    WHERE f.yyyymm//100<2026 AND c.name_ko_mofa IN ('중국','미합중국','일본','호주','베트남')
    GROUP BY 1,2 ORDER BY 2,1
''')
pv = traj.pivot(index='yr', columns='국가', values='bal')
print(pv.to_string())

f=plt.figure(figsize=(8,3.8)); ax=f.gca()
for col in pv.columns: ax.plot(pv.index, pv[col], marker='.', label=L(col))
ax.axhline(0,color='black',lw=.8)
ax.set_ylabel('Trade balance (100M USD)'); ax.set_xlabel('Year')
ax.set_title('Bilateral Trade Balance Over Time (selected partners)')
ax.legend(fontsize=7); ax.grid(alpha=.3); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
f.tight_layout(); plt.show()

In [ ]:
# 대중국 수출·수입·수지 (억달러)
cn = q('''
    SELECT f.yyyymm//100 yr, ROUND(SUM(f.exp_dlr)/1e8,0) 수출, ROUND(SUM(f.imp_dlr)/1e8,0) 수입,
           ROUND(SUM(f.exp_dlr-f.imp_dlr)/1e8,0) 수지
    FROM fact_trade f JOIN dim_country c USING(stat_cd)
    WHERE c.name_ko_mofa='중국' AND f.yyyymm//100<2026 GROUP BY 1 ORDER BY 1
''')
print(cn.to_string(index=False))

f=plt.figure(figsize=(8,3.8)); ax=f.gca()
ax.plot(cn['yr'], cn['수출'], marker='o', ms=3, label='Exports to China')
ax.plot(cn['yr'], cn['수입'], marker='s', ms=3, label='Imports from China')
ax.plot(cn['yr'], cn['수지'], marker='.', color='crimson', label='Balance')
ax.axhline(0,color='gray',lw=.7)
ax.set_ylabel('100M USD'); ax.set_xlabel('Year'); ax.set_title('Korea-China Trade (2007-2025)')
ax.legend(fontsize=7); ax.grid(alpha=.3); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
f.tight_layout(); plt.show()

## 마무리

한국은 **흑자국(157)이 적자국(81)보다 많고**, 흑자 상위는 미국·베트남·홍콩, 적자 상위는 자원공급국(사우디·호주·카타르 등)과 선진 제조국(일본·독일)이다. 가장 큰 변화는 **대중국 수지가 2023년 흑자에서 적자로 반전**한 것. 모든 셀은 관측된 구조를 서술할 뿐 인과를 주장하지 않는다. 한계: 중계무역(홍콩)·편의치적(마셜제도)·최종목적국 기준·EU 제외. 상세는 `chapter03.md` 참조.

In [ ]:
con.close()
print("연결 종료.")